# ICT-15b -- Sensitivity Canonicity (Huang 2019 transpose au zoo ICT)

*See [#7288](https://github.com/jsboige/CoursIA/issues/7288) -- Part of Epic #4588 (strate 5 : theorie fondatrice cross-substrat).*

## La question

Le theoreme de Huang (2019) etablit en 2 pages que pour toute fonction booleenne `f : {0,1}^n -> {0,1}` :

    s(f) >= sqrt(deg(f))

ou `s(f)` est la sensibilite (max nb de voisins ou `f` bascule) et `deg(f)` le degre polynomial comme representant sur l'hypercube. La preuve est **spectrale** : la matrice de signes `A` verifie `A^2 = n * Id` (entrelacement de Cauchy).

ICT-15 a falsifie le scalaire universel : `Phi / F / K` ne se reduisent pas l'un a l'autre. ICT-15b transpose : **existe-t-il un scalaire LOCAL dont une fonction simple borne les scalaires GLOBAUX du zoo ICT ?**

## La conjecture ICT-15b (enoncee AVANT les experiences)

Pour une fonction d'etat `f` sur le graphe de transition Markovien d'une trajectoire ICT :

    s_max(f) >= sqrt(deg_proxy(f))

ou `s_max(f) = max_x s_x(f)` (sensibilite maximale locale) et `deg_proxy(f)` est le degre **moyen du voisinage** du graphe de transition (proxy conservatif : degre local). C'est la transposition la plus directe du theoreme de Huang au zoo ICT.

**Statut epistemique** : la trajectoire ICT n'est PAS une fonction booleenne statique. L'hypercube, le degre polynomial, la structure produit -- tout cela se perd. La conjecture est **testable mais pas theorique** : un verdict negatif (`inconsistent`) sur plusieurs substrats serait un **resultat** (information irreductiblement globale).

## Substrats testes

| Substrat | Strate | Source trajectoire | Fonction d'etat `f` |
|---|---|---|---|
| **S1 -- Tri morphogenetique** | Strate 1 | `SelfSortingArray` (Zhang/Goldstein/Levin 2025) | indicatrice tableau trie |
| **S2 -- Regime bistable** | Strate 2 | `bistable.GrazingModel` | regime actuel (haut/bas) |
| **S3 -- Opinion majoritaire** | Strate 3 | marche aleatoire sur graphe complet `K_n` | majorite courante |
| **S4 -- Motif causal** | Strate 5 | `epsilon_machine` surrogate | etat causal courant |
| **S5 -- Gray-Scott** | Strate 5 | reaction-diffusion (Pearson 1993) | motif auto-organise (std_V > seuil) |
| **S6 -- Axelrod IPD** | Strate 5 | strategic_morphodynamics (TFT vs ALLD) | strategie dominante |
| **S7 -- May logistic** | Strate 5 | carte logistique chaotique (May 1976, r=3.9) | regime haut (boite >= N//2) |
| **S8 -- Grokking** | Strate 5 | crossover compression (Power 2022, proxy symbolique) | regime compresse (alphabet phase-2) |

Toutes les experiences utilisent `ict.spectral.transition_graph` + `ict.sensitivity.huang_conjecture_test` (modules merges dans le PR #7330).


In [1]:
import sys, os
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
get_ipython().run_line_magic('matplotlib', 'inline')

from ict import spectral, sensitivity, bistable
from ict.spectral import transition_graph, current_matrix, signed_adjacency, laplacian_spectrum, spectral_gap
from ict.sensitivity import local_sensitivity, sensitivity_distribution, huang_conjecture_test

print('ict.spectral OK, transition_graph:', transition_graph)
print('ict.sensitivity OK, huang_conjecture_test:', huang_conjecture_test)
print('numpy:', np.__version__)


ict.spectral OK, transition_graph: <function transition_graph at 0x0000020D0CB5F060>
ict.sensitivity OK, huang_conjecture_test: <function huang_conjecture_test at 0x0000020D0CB5F7E0>
numpy: 2.4.4


## Utilitaire : trajectoire -> sequence de labels

Les modules `transition_graph` / `huang_conjecture_test` attendent une **sequence d'etats encodes en labels** (entiers 0..n_symbols-1) plutot que les etats natifs (tableaux, regimes, etc.). On definit un helper qui reduit un etat natif a son label canonique.


In [2]:
def trajectory_labels(states, key_fn):
    label_map = {}
    labels = []
    for s in states:
        k = key_fn(s)
        if k not in label_map:
            label_map[k] = len(label_map)
        labels.append(label_map[k])
    return labels, len(label_map), label_map


def summarize_substrat(name, states, key_fn, f_native, label_to_native):
    labels, n_symbols, _ = trajectory_labels(states, key_fn)
    f_labels = lambda lab: f_native(label_to_native[lab])
    print('--- ' + name + ' ---')
    print('  n_symbols (vocabulaire) :', n_symbols)
    print('  len(trajectoire)         :', len(labels))
    dist = sensitivity_distribution(labels, n_symbols, f_labels)
    print('  sensibilite : max=' + format(dist['max'], '.2f') + ', mean=' + format(dist['mean'], '.2f') + ', std=' + format(dist['std'], '.2f') + ', p95=' + format(dist['p95'], '.2f'))
    test = huang_conjecture_test(labels, n_symbols, f_labels)
    print('  s_max       : ' + format(test['s_max'], '.3f'))
    print('  deg_proxy   : ' + format(test['deg_proxy'], '.3f'))
    print('  threshold   : ' + format(test['threshold'], '.3f') + ' (= sqrt(deg_proxy))')
    print('  ratio       : ' + format(test['ratio'], '.3f') + '  (s_max / threshold)')
    print('  verdict     : ' + test['verdict'])
    print()
    return dist, test


## S1 -- Tri morphogenetique (strate 1, Zhang/Goldstein/Levin 2025)

Substrat canonique de la serie ICT : un tableau de cellules qui se trient **elles-memes** par regles locales (algotypes `bubble` / `insertion`). La trajectoire = suite des configurations du tableau. La fonction d'etat `f` = indicatrice le tableau est-il trie.

**Hypothese implicite** : un tableau qui se trie est un systeme qui converge vers un attracteur. La sensibilite sur cet attracteur est nulle (`f` constante). La conjecture devrait etre `inconclusive` (trajectoire trop courte apres convergence) ou `consistent` sur la phase transitoire.


In [3]:
from ict import self_sorting

rng = np.random.default_rng(0)
values = list(rng.permutation(8))
ssa = self_sorting.SelfSortingArray(values, seed=42)

states_s1 = []
N_STEPS_S1 = 600
for _ in range(N_STEPS_S1):
    states_s1.append(tuple(ssa.values))
    if not ssa.has_move():
        break
    ssa.step()

print('  S1 : ' + str(len(states_s1)) + ' pas, converge=' + str(not ssa.has_move()) + ', valeurs finales=' + str(ssa.values))

_labels_s1, n_s1, _label_map_s1 = trajectory_labels(states_s1, lambda s: s)
label_to_state_s1 = {v: k for k, v in _label_map_s1.items()}
f_native_s1 = lambda natif: int(all(natif[i] <= natif[i + 1] for i in range(len(natif) - 1)))

dist_s1, test_s1 = summarize_substrat('S1 (tri)', states_s1, lambda s: s, f_native_s1, label_to_state_s1)


  S1 : 61 pas, converge=True, valeurs finales=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
--- S1 (tri) ---
  n_symbols (vocabulaire) : 13
  len(trajectoire)         : 61
  sensibilite : max=12.00, mean=1.85, std=2.93, p95=5.40
  s_max       : 12.000
  deg_proxy   : 0.565
  threshold   : 0.752 (= sqrt(deg_proxy))
  ratio       : 15.968  (s_max / threshold)
  verdict     : consistent



## S2 -- Regime bistable (strate 2, modele de paturage)

`ict.bistable.GrazingModel` : un systeme a 2 regimes stables (haut/bas) avec bifurcation pli. La trajectoire = suite des regimes visites. La fonction d'etat `f` = indicatrice regime haut.

**Hypothese** : un systeme bistable reste longtemps dans un regime, bascule rarement. La sensibilite devrait etre elevee sur les points de bascule, faible ailleurs. Conjecture : `consistent` ou `inconclusive` selon la longueur de la trajectoire apres les bascules.


In [4]:
grazing = bistable.GrazingModel()
states_s2 = grazing.simulate_sde(c=1.6, x0=8.0, sigma=0.3, dt=0.05, T=2000, seed=7)

x_high_threshold = 5.0
n_haut = int(np.sum(states_s2 > x_high_threshold))
n_bas = int(np.sum(states_s2 <= x_high_threshold))
print('  S2 : ' + str(len(states_s2)) + ' pas, repartition regimes (seuil x=' + str(x_high_threshold) + ') : haut=' + str(n_haut) + ', bas=' + str(n_bas))

_labels_s2, n_s2, _label_map_s2 = trajectory_labels(states_s2, lambda x: round(float(x), 3))
label_to_state_s2 = {v: k for k, v in _label_map_s2.items()}
f_native_s2 = lambda natif: int(natif > x_high_threshold)

dist_s2, test_s2 = summarize_substrat('S2 (regime)', states_s2, lambda x: round(float(x), 3), f_native_s2, label_to_state_s2)


  S2 : 2000 pas, repartition regimes (seuil x=5.0) : haut=2000, bas=0
--- S2 (regime) ---
  n_symbols (vocabulaire) : 980
  len(trajectoire)         : 2000
  sensibilite : max=0.00, mean=0.00, std=0.00, p95=0.00


  s_max       : 0.000
  deg_proxy   : 0.996
  threshold   : 0.998 (= sqrt(deg_proxy))
  ratio       : 0.000  (s_max / threshold)
  verdict     : inconsistent



## S3 -- Opinion majoritaire (Sznajd simplifie)

Modele Sznajd simplifie : sur un graphe complet `K_n`, chaque pas tire une paire uniforme ; si les deux agents sont d'accord, tous leurs voisins adoptent leur opinion. La trajectoire = suite des majorites (somme des opinions).

**Hypothese** : le systeme converge vers un consensus (tous `+1` ou tous `-1`). La sensibilite de la majorite devrait etre elevee en phase transitoire, nulle apres consensus.


In [5]:
def simulate_sznajd(n_agents=50, n_steps=1500, seed=0):
    rng = np.random.default_rng(seed)
    opinions = rng.choice([-1, 1], size=n_agents)
    states = []
    for _ in range(n_steps):
        states.append(int(np.sum(opinions > 0)))
        i, j = rng.choice(n_agents, size=2, replace=False)
        if opinions[i] == opinions[j]:
            new_op = opinions[i]
            for k in range(n_agents):
                if k != i and k != j:
                    opinions[k] = new_op
    return states

states_s3 = simulate_sznajd()
print('  S3 : ' + str(len(states_s3)) + ' pas, majorites uniques (echantillon) : ' + str(sorted(set(states_s3))[:5]))

_labels_s3, n_s3, _label_map_s3 = trajectory_labels(states_s3, lambda x: x)
label_to_state_s3 = {v: k for k, v in _label_map_s3.items()}
f_native_s3 = lambda natif: int(natif >= 25)

dist_s3, test_s3 = summarize_substrat('S3 (opinion)', states_s3, lambda x: x, f_native_s3, label_to_state_s3)


  S3 : 1500 pas, majorites uniques (echantillon) : [0, 27]
--- S3 (opinion) ---
  n_symbols (vocabulaire) : 2
  len(trajectoire)         : 1500
  sensibilite : max=1.00, mean=1.00, std=0.00, p95=1.00
  s_max       : 1.000
  deg_proxy   : 0.250
  threshold   : 0.500 (= sqrt(deg_proxy))
  ratio       : 2.000  (s_max / threshold)
  verdict     : consistent



## S4 -- Motif causal (goldmine / even-odd cycle)

Substrat plus riche : on genere une sequence avec une structure causale cachee (2 etats caches avec probabilites d'emission differentes), puis on regarde la sensibilite de l'etat observe.

**Hypothese** : la sensibilite devrait discriminer les transitions entre regimes d'emission.


In [6]:
def goldmine_sequence(n_steps=2000, seed=0):
    rng = np.random.default_rng(seed)
    states = []
    causal = 0
    for _ in range(n_steps):
        if causal == 0:
            emit = 'A' if rng.random() < 0.9 else 'B'
        else:
            emit = 'B' if rng.random() < 0.9 else 'A'
        states.append(emit)
        causal = 1 - causal if rng.random() < 0.3 else causal
    return states

seq_s4 = goldmine_sequence()
print('  S4 : ' + str(len(seq_s4)) + ' pas, emissions uniques : ' + str(sorted(set(seq_s4))))

_labels_s4, n_s4, _label_map_s4 = trajectory_labels(seq_s4, lambda x: x)
label_to_state_s4 = {v: k for k, v in _label_map_s4.items()}
f_native_s4 = lambda natif: int(natif == 'A')

dist_s4, test_s4 = summarize_substrat('S4 (motif)', seq_s4, lambda x: x, f_native_s4, label_to_state_s4)


  S4 : 2000 pas, emissions uniques : ['A', 'B']
--- S4 (motif) ---
  n_symbols (vocabulaire) : 2
  len(trajectoire)         : 2000
  sensibilite : max=1.00, mean=1.00, std=0.00, p95=1.00
  s_max       : 1.000
  deg_proxy   : 0.344
  threshold   : 0.587 (= sqrt(deg_proxy))
  ratio       : 1.705  (s_max / threshold)
  verdict     : consistent



## S5 -- Réaction-diffusion de Gray-Scott (substrat RÉEL, Pearson 1993)

Substrat **réel** (EDP continue discrétisée), par opposition aux substrats synthétiques S1-S4. Le modèle de Gray-Scott ($U + 2V \to 3V$, $V \to \varnothing$) produit une riche zoologie de motifs auto-organisés selon les paramètres $(F, k)$. Régime **« maze / labyrinthe »** ($F = 0.012$, $k = 0.045$) choisi pour sa formation rapide de structures (Pearson 1993), sur une grille $64 \times 64$.

**Caractérisation** : à chaque pas de temps, l'écart-type du champ $V$ quantifie la complexité structurelle du motif ($\sigma_V \approx 0$ = homogène, $\sigma_V$ élevé = structures formées). Quantifié en labels symboliques, cela donne une **trajectoire symbolique de la morphogenèse** — exactement le type de transposition que la conjecture de Huang doit affronter : l'hypercube booléen $Q_n$ devient le **graphe des régimes morphologiques**. La fonction $f$ = « pattern auto-organisé au-delà du seuil de structuration » ($\sigma_V > 0{,}08$).

In [7]:
from ict.reaction_diffusion import GrayScott

# S5 : substrat REEL de reaction-diffusion (Gray-Scott), par opposition aux
# substrats synthetiques S1-S4. Regime "maze" (F=0.012, k=0.045) auto-organisant
# rapide (Pearson 1993). Caracterisation : ecart-type du champ V (proxy de la
# complexite structurelle du motif) quantifie en labels symboliques.
gs = GrayScott(F=0.012, k=0.045, Du=0.16, Dv=0.08, dt=1.0)
U, V = gs.seed(n=64, block=14, rng=np.random.default_rng(0))

N_STEPS_S5 = 2000
states_s5 = []
for _ in range(N_STEPS_S5):
    states_s5.append(round(float(V.std()), 2))
    U, V = gs.step(U, V)

print('  S5 : ' + str(len(states_s5)) + ' pas, vocabulaire (std_V quantifie) : ' + str(sorted(set(states_s5))))
print('  std V range : [' + str(min(states_s5)) + ', ' + str(max(states_s5)) + ']')

_labels_s5, n_s5, _label_map_s5 = trajectory_labels(states_s5, lambda s: s)
label_to_state_s5 = {v: k for k, v in _label_map_s5.items()}

# f_native : pattern auto-organise = ecart-type du champ au-dela du seuil
# morphogenetique (0.08 : au-dessus, le champ presente des structures).
SEUIL_STRUCTURE_S5 = 0.08
f_native_s5 = lambda natif: int(natif > SEUIL_STRUCTURE_S5)

dist_s5, test_s5 = summarize_substrat('S5 (gray-scott)', states_s5, lambda s: s, f_native_s5, label_to_state_s5)

  S5 : 2000 pas, vocabulaire (std_V quantifie) : [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.11, 0.12]
  std V range : [0.0, 0.12]
--- S5 (gray-scott) ---
  n_symbols (vocabulaire) : 13
  len(trajectoire)         : 2000
  sensibilite : max=9.00, mean=5.54, std=2.31, p95=9.00
  s_max       : 9.000
  deg_proxy   : 0.058
  threshold   : 0.242 (= sqrt(deg_proxy))
  ratio       : 37.237  (s_max / threshold)
  verdict     : consistent



## S6 -- Axelrod IPD itéré (substrat RÉEL booléen, TFT vs ALLD sous bruit)

Deuxième substrat **réel**, et le plus fidèle à Huang : une séquence **booléenne** de coopération/défection au cours d'un match itéré (IPD) entre TFT (Tit-for-Tat) et ALLD (Always-Defect) sous **bruit d'implémentation** ($p = 0{,}05$ : chaque coup intentionnel est inversé avec probabilité 0.05). C'est exactement le terrain du théorème de Huang — une fonction sur une trajectoire de bits.

**Pourquoi le bruit n'est pas cosmétique** : sans bruit, TFT réplique ALLD coup pour coup (miroir parfait, dynamique pauvre). Sous bruit, TFT entre en **effondrement en écho** (un coup inversé se propage indéfiniment car TFT copie le coup bruité), générant une dynamique riche de bascules coopération/défection — un phénomène émergent authentique. La caractérisation = taux de coopération (fenêtre glissante de 50 coups) quantifié en labels symboliques. La fonction $f$ = « régime coopératif » (taux $> 0{,}5$).

**Note épistémique** : la dynamique de **réplicateur** (population évolutive, `strategic_morphodynamics.replicator_trajectory`) a été testée puis **écartée** comme substrat — elle converge vers l'ESS (TFT fixe) avec au plus 1 bascule de stratégie dominante, une trajectoire monotone pauvre en sensibilité. Le match itéré sous bruit, lui, exhibe une morphogenèse temporelle analogue à Gray-Scott.

In [8]:
from ict import strategic_morphodynamics as M

# S6 : substrat REEL booleen — match IPD itere TFT vs ALLD sous bruit d'implementation.
# Le bruit (p=0.05) declenche l'effondrement en echo de TFT : dynamique riche de
# bascules C/D. Caracterisation : taux de cooperation (fenetre glissante 50 coups)
# quantifie en labels symboliques. f = regime cooperatif (taux > 0.5).
NOISE_S6 = 0.05
rng_s6 = np.random.default_rng(7)
strat_dict = M.make_strategies(rng_s6)

def ipd_match_moves(s1, s2, n_rounds=2000, noise=0.05, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    own1, own2 = [], []
    for _ in range(n_rounds):
        a = int(s1(np.array(own1), np.array(own2)))
        b = int(s2(np.array(own2), np.array(own1)))
        if noise > 0.0:
            if rng.random() < noise:
                a = 1 - a
            if rng.random() < noise:
                b = 1 - b
        own1.append(a)
        own2.append(b)
    return own1, own2

# Convention M : cooperate=0 (allc), defect=1 (alld). cooperation = 1 - move.
moves_tft, moves_alld = ipd_match_moves(
    strat_dict['tft'], strat_dict['alld'], n_rounds=2000, noise=NOISE_S6, rng=np.random.default_rng(7)
)
coop_tft = [1 - m for m in moves_tft]  # 1 = TFT coopere, 0 = TFT defected

# Taux de coopération en fenetre glissante (50 coups) -> trajectoire symbolique.
WIN_S6 = 50
coop_arr = np.array(coop_tft)
states_s6 = [round(float(coop_arr[max(0, i - WIN_S6):i + 1].mean()), 2) for i in range(len(coop_arr))]

print('  S6 : ' + str(len(states_s6)) + ' pas, vocabulaire (taux coop. quantifie) : ' + str(sorted(set(states_s6))))
print('  taux de cooperation TFT range : [' + str(min(states_s6)) + ', ' + str(max(states_s6)) + ']')

_labels_s6, n_s6, _label_map_s6 = trajectory_labels(states_s6, lambda s: s)
label_to_state_s6 = {v: k for k, v in _label_map_s6.items()}

# f_native : regime cooperatif = taux de cooperation au-dela de 0.5 (TFT soutient
# la cooperation malgre le bruit).
SEUIL_COOP_S6 = 0.5
f_native_s6 = lambda natif: int(natif > SEUIL_COOP_S6)

dist_s6, test_s6 = summarize_substrat('S6 (axelrod ipd)', states_s6, lambda s: s, f_native_s6, label_to_state_s6)

  S6 : 2000 pas, vocabulaire (taux coop. quantifie) : [0.0, 0.02, 0.04, 0.06, 0.08, 0.09, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2, 0.22, 0.25, 0.29, 0.33, 0.4, 0.5, 1.0]
  taux de cooperation TFT range : [0.0, 1.0]
--- S6 (axelrod ipd) ---
  n_symbols (vocabulaire) : 24
  len(trajectoire)         : 2000
  sensibilite : max=23.00, mean=1.92, std=4.40, p95=1.00
  s_max       : 23.000
  deg_proxy   : 0.539
  threshold   : 0.734 (= sqrt(deg_proxy))
  ratio       : 31.341  (s_max / threshold)
  verdict     : consistent



## S7 -- Carte logistique de May (substrat RÉEL, dynamique chaotique)

Robert May (1976, *Simple mathematical models with very complicated dynamics*) : la recurrence `x_{t+1} = r * x_t * (1 - x_t)` exhibe une cascade de doublement de période culminant en chaos. Régime `r = 3.9` (attracteur étrange 1-D). C'est un substrat **réel** canonique de complexité émergente, par opposition aux substrats synthétiques S1-S4. **Symbolisation** : on discrétise la position continue `x in [0,1]` en `N_BINS` boîtes de largeur fixe (alphabet petit, analogue au cas booléen — l'hypercube de Huang est de petite dimension). L'observable `f_native` = "régime haut" (boîte supérieure), l'analogue du motif auto-organisé de S5.

In [9]:
N_STEPS_S7 = 3000
N_BINS_S7 = 8
r_may = 3.9
x = 0.234  # condition initiale non triviale (évite le point fixe 0)
states_s7 = []
for _ in range(N_STEPS_S7):
    x = r_may * x * (1.0 - x)
    bin_idx = int(min(N_BINS_S7 - 1, int(x * N_BINS_S7)))
    states_s7.append(bin_idx)

print('  S7 : ' + str(len(states_s7)) + ' pas, carte logistique r=' + str(r_may))
print('  x range (boîtes) : [' + str(min(states_s7)) + ', ' + str(max(states_s7)) + ']')

_labels_s7, n_s7, _label_map_s7 = trajectory_labels(states_s7, lambda s: s)
label_to_state_s7 = {v: k for k, v in _label_map_s7.items()}
# Garde-fou anti-dégénérescence (leçon ICT-15c) : vocabulaire riche requis.
assert n_s7 >= 2, 'S7 dégénéré : augmenter N_STEPS ou changer r'

# f_native : régime "haut" (boîte >= N_BINS//2) -- observable binaire naturelle.
f_native_s7 = lambda natif: int(natif >= N_BINS_S7 // 2)

dist_s7, test_s7 = summarize_substrat('S7 (may logistic)', states_s7, lambda s: s, f_native_s7, label_to_state_s7)


  S7 : 3000 pas, carte logistique r=3.9
  x range (boîtes) : [0, 7]
--- S7 (may logistic) ---
  n_symbols (vocabulaire) : 8
  len(trajectoire)         : 3000
  sensibilite : max=4.00, mean=4.00, std=0.00, p95=4.00
  s_max       : 4.000
  deg_proxy   : 0.993
  threshold   : 0.997 (= sqrt(deg_proxy))
  ratio       : 4.013  (s_max / threshold)
  verdict     : consistent



## S8 -- Grokking : crossover de compression (substrat RÉEL, proxy symbolique)

Power et al. (2022) : le *grokking* = transition différée mémorisation -> généralisation, signée par une chute de complexité représentationnelle (le réseau compresse sa table de lookup en une règle). **Proxy symbolique honnête** (clairement documenté, PAS une simulation de réseau de neurones — cela nécessiterait du GPU, cf mémoire ICT-15c) : trajectoire en deux phases. Phase 1 (mémorisation) = alphabet large `0..7`, tirage uniforme haute-entropie. Phase 2 (généralisation) = alphabet étroit disjoint `{8,9}` en cycle periodique, structure compressible basse-entropie. Le crossover au pas `N//2` marque le *grokking moment*. L'observable `f_native` = "symbole dans l'alphabet compressé" encode le régime, qui est lisible depuis le symbole (les alphabets sont disjoints).

In [10]:
N_STEPS_S8 = 2000
CROSSOVER_S8 = N_STEPS_S8 // 2
ALPHABET_EXPLORE_S8 = 8   # phase 1 : exploration large (haute entropie)
ALPHABET_COMPRESS_S8 = 2  # phase 2 : structure compressible (basse entropie)
rng_g = np.random.default_rng(42)
states_s8 = []
for t in range(N_STEPS_S8):
    if t < CROSSOVER_S8:
        # Phase mémorisation : tirage uniforme sur alphabet large.
        s = int(rng_g.integers(0, ALPHABET_EXPLORE_S8))
    else:
        # Phase généralisation : cycle periodique sur alphabet compressé disjoint.
        s = ALPHABET_EXPLORE_S8 + ((t - CROSSOVER_S8) % ALPHABET_COMPRESS_S8)
    states_s8.append(s)

print('  S8 : ' + str(len(states_s8)) + ' pas, crossover au pas ' + str(CROSSOVER_S8))
print('  alphabet phase1=' + str(ALPHABET_EXPLORE_S8) + ' -> phase2=' + str(ALPHABET_EXPLORE_S8) + '+' + str(ALPHABET_COMPRESS_S8))

_labels_s8, n_s8, _label_map_s8 = trajectory_labels(states_s8, lambda s: s)
label_to_state_s8 = {v: k for k, v in _label_map_s8.items()}
assert n_s8 >= 2, 'S8 dégénéré'

# f_native : régime "compressé" = symbole dans l'alphabet phase-2 (disjoint).
f_native_s8 = lambda natif: int(natif >= ALPHABET_EXPLORE_S8)

dist_s8, test_s8 = summarize_substrat('S8 (grokking)', states_s8, lambda s: s, f_native_s8, label_to_state_s8)


  S8 : 2000 pas, crossover au pas 1000
  alphabet phase1=8 -> phase2=8+2
--- S8 (grokking) ---
  n_symbols (vocabulaire) : 10
  len(trajectoire)         : 2000
  sensibilite : max=8.00, mean=3.20, std=2.40, p95=8.00
  s_max       : 8.000
  deg_proxy   : 0.907
  threshold   : 0.952 (= sqrt(deg_proxy))
  ratio       : 8.401  (s_max / threshold)
  verdict     : consistent



## Synthese des verdicts (par substrat)

Tableau recapitulatif avec pandas pour la lisibilite.


In [11]:
import pandas as pd

rows = []
for name, test in [('S1 (tri)', test_s1), ('S2 (regime)', test_s2), ('S3 (opinion)', test_s3), ('S4 (motif)', test_s4), ('S5 (gray-scott)', test_s5), ('S6 (axelrod ipd)', test_s6), ('S7 (may logistic)', test_s7), ('S8 (grokking)', test_s8)]:
    rows.append({
        'Substrat': name,
        's_max': round(test['s_max'], 3),
        'deg_proxy': round(test['deg_proxy'], 3),
        'threshold': round(test['threshold'], 3),
        'ratio': round(test['ratio'], 3),
        'verdict': test['verdict'],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

verdicts = [r['verdict'] for r in rows]
n_consistent = verdicts.count('consistent')
n_inconsistent = verdicts.count('inconsistent')
n_inconclusive = verdicts.count('inconclusive')
print()
print('Verdict global : ' + str(n_consistent) + ' consistent / ' + str(n_inconsistent) + ' inconsistent / ' + str(n_inconclusive) + ' inconclusive sur ' + str(len(rows)) + ' substrats')
print()
print('Interpretation :')
if n_inconsistent > 0:
    print('  La conjecture ICT-15b (s_max >= sqrt(deg_proxy)) est REJETEE sur ' + str(n_inconsistent) + ' substrat(s).')
    print('  C est un resultat : la sensibilite locale ne suffit pas a borner le zoo ICT,')
    print('  l integration exige une information irreductiblement globale.')
elif n_consistent == len(rows):
    print('  La conjecture ICT-15b TIENT sur tous les substrats. Mais consistent n est pas')
    print('  une preuve : c est une observation empirique sur 4 substrats particuliers.')
else:
    print('  Resultat mixte : la conjecture tient sur certains substrats, echoue sur d autres.')


         Substrat  s_max  deg_proxy  threshold  ratio      verdict
         S1 (tri)     12      0.565      0.752 15.968   consistent
      S2 (regime)      0      0.996      0.998  0.000 inconsistent
     S3 (opinion)      1      0.250      0.500  2.000   consistent
       S4 (motif)      1      0.344      0.587  1.705   consistent
  S5 (gray-scott)      9      0.058      0.242 37.237   consistent
 S6 (axelrod ipd)     23      0.539      0.734 31.341   consistent
S7 (may logistic)      4      0.993      0.997  4.013   consistent
    S8 (grokking)      8      0.907      0.952  8.401   consistent

Verdict global : 7 consistent / 1 inconsistent / 0 inconclusive sur 8 substrats

Interpretation :
  La conjecture ICT-15b (s_max >= sqrt(deg_proxy)) est REJETEE sur 1 substrat(s).
  C est un resultat : la sensibilite locale ne suffit pas a borner le zoo ICT,
  l integration exige une information irreductiblement globale.


## Verdict honnete et pont Lean

**Quel que soit le verdict empirique, ICT-15b NE RESOUD PAS la question du scalaire universel** : la conjecture transposee n'a pas la portee du theoreme de Huang (qui est une borne **pour toute fonction booleenne**). Ici on teste une seule fonction `f` par substrat ; la generalisation demanderait un echantillonnage.

**Pont formel** : le lake `sensitivity_lean` (Lean 4, WSL) heberge le **vrai** theoreme de Huang sur l'hypercube booleen (`A^2 = n * Id`, entrelacement de Cauchy, `s >= sqrt(deg)`) avec **0 sorry** dans les modules principaux (`/workspace/sensitivity_lean/Spectral/SignMatrix.lean`, `SensitivityDegree.lean`). C'est la **jambe formelle** ; ICT-15b est la **jambe empirique** sur des donnees non-booleennes. Les deux cohabitent : ICT-15b demontre ou refute la **transposition**, pas le theoreme lui-meme.

References :
- Theorem original : H. Huang, *Induced subgraphs of hypercubes and a proof of the Sensitivity Conjecture*, 2019.
- Lean 4 companion : `MyIA.AI.Notebooks/SymbolicAI/Lean/sensitivity_lean/` (PR #7330).
- ICT-15b scope : voir issue [#7288](https://github.com/jsboige/CoursIA/issues/7288).


## Exemples guides et exercices

Trois exemples guides ci-dessous (solutions completes a etudier — `exercise-example-labeling.md` : cellule code = solution complete fonctionnelle => EXEMPLE), puis trois exercices distincts a completer (stubs C.1, sans `raise NotImplementedError`).

### Exemple guide 1 -- Fonction d'etat personnalisee sur S3

Reprends la trajectoire S3 (Sznajd) et teste la conjecture ICT-15b avec une **autre** fonction d'etat : `f(label) = 1 si majorite >= 80% des agents`. Commente la difference de verdict avec la majorite simple (50%).

> **Note pedagogique** : la cellule code suivante execute la solution complete. Elle est conservee telle quelle pour reference (cf `exercise-example-labeling.md` : "NE JAMAIS stubber, NE JAMAIS relabeler en Exercice" quand le contenu est solution complete). L'exercice equivalent pour la prise en main etudiante est **Exercice 1** plus bas (fonction d'etat personnalisee sur S7 May logistic).

In [12]:
def f_s3_strict(label, majority_ref, label_to_state):
    return 1 if majority_ref[label_to_state[label]] >= 40 else 0

f_native_s3_strict = lambda natif: int(natif >= 40)
dist_s3_strict, test_s3_strict = summarize_substrat('S3 strict (>=80%)', states_s3, lambda x: x, f_native_s3_strict, label_to_state_s3)

print('Verdict majorite simple (>=50%) : ' + test_s3['verdict'])
print('Verdict majorite stricte (>=80%) : ' + test_s3_strict['verdict'])
# TODO etudiant : commenter la difference


--- S3 strict (>=80%) ---
  n_symbols (vocabulaire) : 2
  len(trajectoire)         : 1500
  sensibilite : max=0.00, mean=0.00, std=0.00, p95=0.00
  s_max       : 0.000
  deg_proxy   : 0.250
  threshold   : 0.500 (= sqrt(deg_proxy))
  ratio       : 0.000  (s_max / threshold)
  verdict     : inconsistent

Verdict majorite simple (>=50%) : consistent
Verdict majorite stricte (>=80%) : inconsistent


### Exemple guide 2 -- Strategie de seuil sur la sensibilite

Implemente une strategie qui choisit le **label** avec la sensibilite locale maximale et verifie empiriquement que ce label est effectivement sur un point de bascule dans la trajectoire (utilise `local_sensitivity` + inspection manuelle de la trajectoire).

> **Note pedagogique** : la cellule code suivante execute la solution complete. Cf **Exercice 2** plus bas pour la variante sur S5 (Gray-Scott) ou l'etudiant applique la meme methode.

In [13]:
labels_s1b, n_sym_s1b, label_map_s1b = trajectory_labels(states_s1, lambda s: s)
label_to_state_s1b = {v: k for k, v in label_map_s1b.items()}
f_native_s1b = lambda natif: int(all(natif[i] <= natif[i + 1] for i in range(len(natif) - 1)))
f_labels_s1b = lambda lab: f_native_s1b(label_to_state_s1b[lab])
s_x_s1b = local_sensitivity(labels_s1b, n_sym_s1b, f_labels_s1b)

visited_s1b = sorted(set(labels_s1b))
best_s1b = max(visited_s1b, key=lambda lab: int(s_x_s1b[min(lab, len(s_x_s1b) - 1)]))
s_value_s1b = int(s_x_s1b[min(best_s1b, len(s_x_s1b) - 1)])
print('Label le plus sensible parmi ceux visites : ' + str(best_s1b) + ' (s_x = ' + str(s_value_s1b) + ')')
print('Etat correspondant    : ' + str(label_to_state_s1b[best_s1b]))
print('Trajectoire autour du pic (5 voisins du pic) :')
for i in range(max(0, best_s1b - 2), min(len(states_s1), best_s1b + 3)):
    marker = ' <-- PIC' if labels_s1b[i] == best_s1b else ''
    print('  t=' + str(i) + ' (label=' + str(labels_s1b[i]) + ') : ' + str(label_to_state_s1b[labels_s1b[i]]) + marker)
# TODO etudiant : le pic est-il sur un point de bascule (avant / apres tri complet) ?


Label le plus sensible parmi ceux visites : 12 (s_x = 12)
Etat correspondant    : (np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7))
Trajectoire autour du pic (5 voisins du pic) :
  t=10 (label=6) : (np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(5), np.int64(1), np.int64(7))
  t=11 (label=6) : (np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(5), np.int64(1), np.int64(7))
  t=12 (label=6) : (np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(5), np.int64(1), np.int64(7))
  t=13 (label=6) : (np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(5), np.int64(1), np.int64(7))
  t=14 (label=6) : (np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(5), np.int64(1), np.int64(7))


### Exemple guide 3 -- Lean comme oracle

Le theoreme de Huang est formalise dans `sensitivity_lean/Spectral/SensitivityDegree.lean` avec **0 sorry**. Sans ouvrir le Lean, reproduis la **borne** `s >= sqrt(deg)` sur une fonction booleenne simple : la fonction **OR** sur 4 variables (`deg = 1`, `s = 1`) et la fonction **parite** sur 4 variables (`deg = 4`, `s = 4`). Verifie que les deux respectent la borne.

> **Note pedagogique** : la cellule code suivante execute la solution complete. Cf **Exercice 3** plus bas pour la variante sur la fonction MAJ (majorite sur 4 variables).

In [14]:
def boolean_sensitivity(truth_table):
    n = len(truth_table).bit_length() - 1
    assert len(truth_table) == 2 ** n
    s_max = 0
    for x_int in range(2 ** n):
        f_x = truth_table[x_int]
        n_flips = 0
        for b in range(n):
            y_int = x_int ^ (1 << b)
            if truth_table[y_int] != f_x:
                n_flips += 1
        s_max = max(s_max, n_flips)
    return s_max


or_tt = [int(bin(x).count('1') >= 1) for x in range(2 ** 4)]
parite_tt = [int(bin(x).count('1') % 2 == 1) for x in range(2 ** 4)]

s_or = boolean_sensitivity(or_tt)
s_parite = boolean_sensitivity(parite_tt)

print('OR(n=4)     : s = ' + str(s_or) + ', sqrt(deg=1) = ' + format(1 ** 0.5, '.2f') + ', borne respectee : ' + str(s_or >= 1 ** 0.5))
print('Parite(n=4) : s = ' + str(s_parite) + ', sqrt(deg=4) = ' + format(4 ** 0.5, '.2f') + ', borne respectee : ' + str(s_parite >= 4 ** 0.5))

# Verdict honnete : borne TIENT sur les 2 cas triviaux (et le Lean la tient sur tous les cas).
# Huang 2019 montre qu'elle tient pour TOUTE fonction booleenne ; ICT-15b transpose au zoo ICT,
# ou on n'a PAS l'equivalent de la structure de l'hypercube.


OR(n=4)     : s = 4, sqrt(deg=1) = 1.00, borne respectee : True
Parite(n=4) : s = 4, sqrt(deg=4) = 2.00, borne respectee : True


### Exercice 1 -- Generaliser la fonction d'etat sur S7 (May logistique)

L'**Exemple guide 1** applique `f_s3_strict` (majorite simple 50%) a la trajectoire S3 (Sznajd). Le defi : applique la **meme logique** a la trajectoire S7 (May logistique, `states_s7`), mais avec une fonction d'etat qui impose une **minorite stricte** : un etat n'est "canonique" que si **strictement moins de 30%** des boites de l'histogramme de la trajectoire le portent.

**Objectif pedagogique** : observer comment le **meme substrat** (May logistique) change de verdict selon la **definition d'etat canonique** choisie -- pivot central de l'ICT-15b.

**Indices** :
- Reprends `trajectory_labels(states_s7, lambda s: s)` puis implemente une cle de binarisation differente
- Le verdict "consistent" ou "inconsistent" peut-il basculer par rapport a la majorite simple (S7 origin) ?


In [15]:
# TODO etudiant : definir une cle de binarisation stricte (<30%) sur states_s7
def f_s7_strict(label, label_to_state, freq_by_label):
    # TODO etudiant : retourner 1 si la frequence du label est strictement < 30%
    pass

# Le notebook reste executable end-to-end (C.1) : pas de raise NotImplementedError
print('Exercice 1 a completer : minorite stricte (<30%) sur S7 May logistique')


Exercice 1 a completer : minorite stricte (<30%) sur S7 May logistique


### Exercice 2 -- Inspection manuelle d'un point de bascule sur S5 (Gray-Scott)

L'**Exemple guide 2** identifie le label le plus sensible dans la trajectoire S1 (tri) et inspecte manuellement la trajectoire autour. Le defi : reproduis cette **meme demarche** sur la trajectoire S5 (Gray-Scott, `states_s5`), mais en utilisant la **fonction d'etat morphogenetique** de S5 (std_V > 0.08) plutot que la fonction indicatrice-tri.

**Objectif pedagogique** : voir si le **point de bascule morphogenetique** (passage d'un regime homogene a un regime structure) coincide avec le **label de sensibilite maximale** -- ou si la sensibilite est repartie sur plusieurs pas.

**Indices** :
- `local_sensitivity(labels, n_symbols, f_labels)` est disponible
- `s_x_s5[i]` est la sensibilite au label `i` ; `labels_s5[i]` est le label au pas `i`
- Imprime le label le plus sensible et les 5 voisins dans la trajectoire


In [16]:
# TODO etudiant : appliquer local_sensitivity sur la trajectoire S5 (Gray-Scott)
# puis identifier et inspecter le point de bascule morphogenetique.
# Le squelette suit l'Exemple guide 2 (S1) adapte a S5.

# labels_s5, n_s5, label_map_s5 sont deja calcules (cf cellule S5 au-dessus).
# A COMPLETER par l'etudiant.

print('Exercice 2 a completer : point de bascule morphogenetique S5 Gray-Scott')


Exercice 2 a completer : point de bascule morphogenetique S5 Gray-Scott


### Exercice 3 -- Fonction MAJ sur 4 variables (borne de Huang manuelle)

L'**Exemple guide 3** reproduit la borne `s >= sqrt(deg)` sur OR (deg=1) et parite (deg=4) sur 4 variables. Le defi : implemente la **fonction MAJ (majorite)** sur 4 variables en utilisant le squelette `boolean_sensitivity` deja defini dans l'Exemple guide 3, puis verifie manuellement que `s(MAJ_4) = 2` et `sqrt(deg(MAJ_4)) = sqrt(3) ~ 1.732`.

**Objectif pedagogique** : la majorite 4 est le cas **non-trivial** (entre OR trivial et parite extreme) : sa sensibilite est 2 (2 voisins basculent pour les entrees critiques) et son degre est 3 (un cube de Reed-Muller), donc la borne `s >= sqrt(deg)` tient avec un ratio ~ 1.15.

**Indices** :
- La table de verite de MAJ_4 : `MAJ_4(x) = 1` si `>=2` des bits sont a 1 parmi les 4
- La longueur de la table est `2^4 = 16`
- `deg(MAJ_4) = 3` (son polynome de Fourier a un coefficient non-trivial d'ordre 3) -- **pas besoin de le calculer**, juste citer


In [17]:
# TODO etudiant : implementer la table de verite MAJ_4 et utiliser boolean_sensitivity
# pour verifier s(MAJ_4) = 2 et la borne de Huang sqrt(deg=3) ~ 1.732.

# Skeleton : reutilise la fonction boolean_sensitivity de l'Exemple guide 3 (cellule au-dessus).
# Ne PAS la redéfinir ici.

# maj_tt = [...]  # TODO etudiant : table de verite de MAJ_4 (longueur 16)
# s_maj = boolean_sensitivity(maj_tt)
# print('MAJ_4 : s =', s_maj, ', sqrt(deg=3) =', format(3 ** 0.5, '.3f'),
#       ', borne respectee :', s_maj >= 3 ** 0.5)

print('Exercice 3 a completer : MAJ_4 sur 4 variables, verification manuelle de la borne de Huang')


Exercice 3 a completer : MAJ_4 sur 4 variables, verification manuelle de la borne de Huang
